In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lyria 3 Music Generation

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Faudio%2Fmusic%2Fgetting-started%2Flyria3_music_generation.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/audio/music/getting-started/lyria3_music_generation.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/music/getting-started/lyria3_music_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author |
| --- |
| [Katie Nguyen](https://github.com/katiemn) |

## Overview

### Lyria 3

Lyria 3 on Agent Platform gives application developers access to Google's cutting-edge music generation. This model creates high-fidelity clips, tracks, and music streams across global languages and genres.

In this tutorial, you will learn how to use the Google Gen AI SDK for Python to interact with Lyria 3 and generate new music, including:
- Full tracks with vocals from text prompts
- Clips from image prompts
- Custom lyrics
- Music streaming in multiple languages with the Interactions API

## Get started

### Install Google Gen AI SDK for Python

In [ ]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the following cell to authenticate your environment.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [3]:
import base64
import os
import re

from IPython.display import Audio, Image, Markdown, display
from google import genai
from google.genai import types

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [4]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)

### Define a helper function to play audio

In [5]:
def display_output(response):
    for part in response.parts:
        if part.text:
            text = part.text

            # Remove section tags, add new lines
            text = re.sub(r"\[\[.*?\]\]", "", text)
            text = re.sub(
                r"\s*\[(\d+\.\d+)(?::?(\d+\.\d+))?:?\]:?", r"\n\n[\1\2] ", text
            )
            text = text.replace("[:]", "\n\n")
            clean_text = text.strip()

            display(Markdown(clean_text))
        if part.inline_data:
            display(Audio(data=part.inline_data.data, autoplay=False))

### Load the music generation models

In [6]:
music_clip_model = "lyria-3-clip-preview"
music_model = "lyria-3-pro-preview"

## Generate music
Now, you'll generate music from text and/or image prompts. With Lyria 3 you can choose to generate 30-second clips with the `lyria-3-clip-preview` model or generate tracks up to 3 minutes with `lyria-3-pro-preview`.

### Create tracks from text prompts

When prompting Lyria 3, it's helpful to consider a few topics:
  - **Style/Genre:** Consider options such as: classical, electronic, rock, jazz, hip hop, or pop. You can even describe more general styles that include cinematic, ambient, or lo-fi.
  - **Vocals:** If you'd like to include vocals, describe a vocal style specifying attributes like vocal range and tone.
  - **Instruments:** Add specific instruments such as a piano, synthesizer, acoustic guitar, drums, strings, or flute.

When parsing the model response, you'll receive text parts containing generated lyrics or information about the song structure in addition to audio bytes in `inline_data`. In order to receive both text and audio outputs, make sure to set `response_modalities` to `['AUDIO', 'TEXT']`.

By default, all music generated with Lyria 3 utilizes [SynthID](https://deepmind.google/technologies/synthid/) and [C2PA](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/content-credentials).

In [ ]:
prompt = """
Sophisticated, rhythmic, and aspirational track with crisp 808 percussion, digital plucks,
and muted electric guitar rhythmic strums. Include breathy, airy Alto female vocal textures
with melodic, minimalist oohs and aahs with heavy reverb and rhythmic delay.
"""

response = client.models.generate_content(
    model=music_model,
    contents=prompt,
    config=types.GenerateContentConfig(response_modalities=["AUDIO", "TEXT"]),
)
display_output(response)

### Generate music clips from images

In this next section, you'll generate a 30-second music clip from an input image. You'll start by downloading and displaying an image in the next cell.

In [ ]:
!wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/flowers.png

input_image = "flowers.png"
display(Image(input_image, width=400))

Now, you'll send a request with a text prompt describing the type of music you'd like generated in reference to the provided image data. While this request only contains one image, you can supply up to 10 images in a single request.

In [ ]:
with open(input_image, "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=music_clip_model,
    contents=[
        types.Part.from_bytes(
            data=image,
            mime_type="image/png",
        ),
        "Generate an instrumental clip based on this input image that starts slowly and builds in intensity.",
    ],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "AUDIO"],
    ),
)

display_output(response)

### Prompting with lyrics

Now, in addition to an input image, you'll include specific lyrics you'd like generated in the final result. Again, start by downloading and displaying the starting input image.

In [ ]:
!wget -q https://storage.googleapis.com/cloud-samples-data/generative-ai/image/dog-ad-2.png

input_image = "dog-ad-2.png"
display(Image(input_image, width=400))

Craft a new prompt with the specific lyrics you'd like to include. You can also specify additional information such as tempo, genre, style, vocal profile, instruments, or other general instructions.

In [ ]:
with open(input_image, "rb") as f:
    image = f.read()

genre_lyrics = """
Genre: Upbeat, acoustic Folk-Pop with a warm and cuddly vibe. Bright acoustic guitars, a soft shaker rhythm, and a friendly, melodic vocal.

Lyrics:
Tail wags and a heavy head,
Time to curl up in your favorite bed.
Soft as a cloud, a dream come true,
The perfect spot for a dog like you.
"""

response = client.models.generate_content(
    model=music_clip_model,
    contents=[
        types.Part.from_bytes(
            data=image,
            mime_type="image/png",
        ),
        genre_lyrics,
    ],
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "AUDIO"],
    ),
)

display_output(response)

### Interactions API

You can also use Lyria 3 with the Interactions API, which is a unified interface for interacting with models and agents.

**Note:** This model can generate music in the following languages - English, German, Spanish, French, Hindi, Japanese, Korean, and Portuguese. You can either write the prompt in one of these languages, or ask for it specifically if writing in a different language.

In [12]:
def display_interaction_output(interaction):
    outputs = getattr(interaction, "outputs", []) or []

    for output in outputs:
        is_dict = isinstance(output, dict)
        output_type = output.get("type") if is_dict else getattr(output, "type", None)

        if output_type == "text":
            text = output.get("text", "") if is_dict else getattr(output, "text", "")
            if not text:
                continue

            # Remove section tags, add new lines
            text = re.sub(r'\[\[.*?\]\]', '', text)
            text = re.sub(r'\s*\[(\d+\.\d+)(?::?(\d+\.\d+))?:?\]:?', r'\n\n[\1\2] ', text)
            text = text.replace('[:]', '\n\n')
            clean_text = text.strip()
            display(Markdown(clean_text))

        elif output_type == "audio":
            data = output.get("data") if is_dict else getattr(output, "data", None)
            if not data:
                continue

            audio_data = base64.b64decode(data)
            display(Audio(data=audio_data, autoplay=False))

In [ ]:
interaction = client.interactions.create(
    model=music_model,
    input="Genera un tema pop",
)
display_interaction_output(interaction)

#### Music streaming

Within the Interactions API, you can generate a content stream by setting `stream` to True. This will output content such as lyrics and descriptions as they're returned from the model, rather than waiting for the request to complete.

In [ ]:
stream = client.interactions.create(
    model=music_model,
    input="Generate a song about spending a day in Seoul in Korean.",
    stream=True,
)

for event in stream:
    if event.event_type == "content.delta":
        delta_dict = event.delta if isinstance(event.delta, dict) else getattr(event, "delta", {})
        if "text" in delta_dict:
            text = delta_dict["text"]
            text = text.replace("[:]", "\n[:]")
            text = re.sub(r"(\[\d+\.\d+:\])", r"\n\1", text)
            display(Markdown(text))
        elif "data" in delta_dict and "audio" in delta_dict.get("mime_type", ""):
            display(Audio(data=base64.b64decode(delta_dict["data"]), autoplay=False))